In [ ]:
!pip install -q torch_geometric scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [ ]:
import torch
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T

dataset = Planetoid( root="./data", name="Cora",
                    transform=T.NormalizeFeatures())
data = dataset[0]
torch.save(data, "cora_graph.pt")
print("Saved Dataset")

Saved Dataset


In [ ]:
import torch

data = torch.load("cora_graph.pt", map_location="cpu",
                  weights_only=False)
print(data)
print("Nodes:", data.num_nodes)
print("Edges:", data.edge_index.shape[1])
print("Features:", data.x.shape)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Nodes: 2708
Edges: 10556
Features: torch.Size([2708, 1433])


In [ ]:
split_transform = T.RandomLinkSplit( num_val=0.05,num_test=0.10,
            is_undirected=True, add_negative_train_samples=False)

train_data, val_data, test_data = split_transform(data)
print(train_data)
print(train_data.edge_label_index[0])
print(train_data.edge_label_index[1])



Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], edge_label=[4488], edge_label_index=[2, 4488])
tensor([910, 492, 682,  ...,  59, 146, 512])
tensor([2085, 1257,  973,  ..., 1725, 1927,  756])


In [ ]:
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)
all_original_edges = data.edge_index.to(device)

In [ ]:
class GCNLinkPredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(1433, 128)
        self.conv2 = GCNConv(128, 64)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x,p=0.3)
        z = self.conv2(x, edge_index)
        return z

    def decode(self, z, edge_label_index):
        source = edge_label_index[0]
        destination = edge_label_index[1]
        scores = (z[source] * z[destination]).sum(dim=-1)
        return scores

In [ ]:
model = GCNLinkPredictor().to(device)
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)
# loss_fxn = nn.BCELoss()
criterion = nn.BCEWithLogitsLoss()
print(model)

GCNLinkPredictor(
  (conv1): GCNConv(1433, 128)
  (conv2): GCNConv(128, 64)
)


In [ ]:
def train():
    model.train()
    optimizer.zero_grad()
    # Generate embedding for train dataset
    z = model.encode(train_data.x,train_data.edge_index)
    pos_edge_index = train_data.edge_label_index
    pos_scores = model.decode(z,pos_edge_index)
    #virtually connecting each node as given
    neg_edge_index = negative_sampling(
        edge_index=all_original_edges,
        num_nodes=data.num_nodes,
        num_neg_samples=pos_edge_index.size(1),
        method="sparse"
    )
    neg_scores = model.decode(z,neg_edge_index)
    scores = torch.cat([pos_scores,neg_scores])
    labels = torch.cat([torch.ones_like(pos_scores),
                        torch.zeros_like(neg_scores)])
    loss = criterion(scores, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
@torch.no_grad()
def evaluate(split_data):
    model.eval()
    z = model.encode(split_data.x,split_data.edge_index)
    scores = model.decode(z,split_data.edge_label_index)
    probabilities = torch.sigmoid(scores)
    labels = split_data.edge_label
    y_true = labels.cpu().numpy()
    y_score = probabilities.cpu().numpy()
    auc = roc_auc_score(y_true,y_score)
    ap = average_precision_score( y_true, y_score)
    return auc, ap

In [ ]:
epochs = 100
for epoch in range(1, epochs + 1):
    loss = train()
    val_auc, val_ap = evaluate(val_data)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:03d} Loss: {loss:.4f}"
              f"Val AUC: {val_auc:.4f} Val AP: {val_ap:.4f}")
test_auc, test_ap = evaluate(test_data)
print("\nFinal Test Results")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"Test Average Precision: {test_ap:.4f}")

Epoch 010 Loss: 0.6849 Val AUC: 0.6889 Val AP: 0.7070
Epoch 020 Loss: 0.6382 Val AUC: 0.7248 Val AP: 0.7437
Epoch 030 Loss: 0.5245 Val AUC: 0.7951 Val AP: 0.7901
Epoch 040 Loss: 0.5102 Val AUC: 0.8323 Val AP: 0.8466
Epoch 050 Loss: 0.4869 Val AUC: 0.8563 Val AP: 0.8709
Epoch 060 Loss: 0.4704 Val AUC: 0.8841 Val AP: 0.8911
Epoch 070 Loss: 0.4583 Val AUC: 0.8952 Val AP: 0.8976
Epoch 080 Loss: 0.4507 Val AUC: 0.8910 Val AP: 0.8971
Epoch 090 Loss: 0.4442 Val AUC: 0.9007 Val AP: 0.9023
Epoch 100 Loss: 0.4476 Val AUC: 0.8954 Val AP: 0.8989

Final Test Results
Test ROC-AUC: 0.9059
Test Average Precision: 0.9136
